In [0]:
base_path = "/Volumes/workspace/default/retail_volume"

bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"

print(bronze_path)
print(silver_path)

/Volumes/workspace/default/retail_volume/bronze
/Volumes/workspace/default/retail_volume/silver


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

print("Bronze Rows:", bronze_df.count())

Bronze Rows: 42481


In [0]:
from pyspark.sql.functions import col

silver_df = (
    bronze_df
    .filter(col("CustomerID").isNotNull())
    .filter(col("Quantity") > 0)
    .filter(col("UnitPrice") > 0)
)

In [0]:
silver_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)



In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

silver_df = (
    bronze_df
    .withColumn("Quantity", col("Quantity").cast("double"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .filter(col("CustomerID").isNotNull())
    .filter(col("Quantity") > 0)
    .filter(col("UnitPrice") > 0)
    .withColumn("Revenue", col("Quantity") * col("UnitPrice"))
)

display(silver_df.select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "Revenue",
    "CustomerID",
    "Country"
).limit(10))

InvoiceNo,StockCode,Description,Quantity,UnitPrice,Revenue,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2.55,15.299999999999999,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6.0,3.39,20.34,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2.75,22.0,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,3.39,20.34,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,3.39,20.34,17850.0,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2.0,7.65,15.3,17850.0,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6.0,4.25,25.5,17850.0,United Kingdom
536366,22633,HAND WARMER UNION JACK,6.0,1.85,11.100000000000001,17850.0,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6.0,1.85,11.100000000000001,17850.0,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32.0,1.69,54.08,13047.0,United Kingdom


In [0]:
silver_path = "/Volumes/workspace/default/retail_volume/silver"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Silver table saved successfully")

Silver table saved successfully


In [0]:
saved_silver_df = spark.read.format("delta").load(silver_path)

print("Silver Rows:", saved_silver_df.count())

display(saved_silver_df.limit(10))

Silver Rows: 26157


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.299999999999999
536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.0
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
536365,22752,SET 7 BABUSHKA NESTING BOXES,2.0,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.3
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,25.5
536366,22633,HAND WARMER UNION JACK,6.0,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.100000000000001
536366,22632,HAND WARMER RED POLKA DOT,6.0,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.100000000000001
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32.0,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,54.08
